In [49]:
import os.path as osp
from collections import defaultdict
from typing import Dict

import numpy as np
import torch
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info
from transformers import (
    AutoProcessor,
    EvalPrediction,
    GenerationConfig,
    Qwen3VLForConditionalGeneration,
    Seq2SeqTrainingArguments,
)

In [50]:
video_path = "datasets/ds2/shots/shot_855/video_cam0-full.mp4"

In [51]:
model_name = "Qwen/Qwen3-VL-4B-Instruct"

# model = Qwen3VLForConditionalGeneration.from_pretrained(
#     model_name,
#     dtype="bfloat16",
#     # device_map="auto",
#     attn_implementation="flash_attention_2",
#     local_files_only=True,
# )
processor = AutoProcessor.from_pretrained(
    model_name, local_files_only=True, use_fast=True
)
tokenizer = processor.tokenizer

Offline mode: forcing local_files_only=True
Offline mode: forcing local_files_only=True
loading configuration file preprocessor_config.json from cache at /scratch/scottc/cache/hf/hub/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17/preprocessor_config.json
loading configuration file preprocessor_config.json from cache at /scratch/scottc/cache/hf/hub/models--Qwen--Qwen3-VL-4B-Instruct/snapshots/ebb281ec70b05090aa6165b016eac8ec08e71b17/preprocessor_config.json
Image processor Qwen2VLImageProcessorFast {
  "crop_size": null,
  "data_format": "channels_first",
  "default_to_square": true,
  "device": null,
  "disable_grouping": null,
  "do_center_crop": null,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_pad": null,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "Qwen2VLImageProcessorFast",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "input_data_format": null,
  "max_

In [52]:
import os

# Constants from qwen_vl_utils (matching vision_process.py)
SPATIAL_MERGE_SIZE = 2
IMAGE_MIN_TOKEN_NUM = 4
IMAGE_MAX_TOKEN_NUM = 16384
# per-frame min/max token count
VIDEO_MIN_TOKEN_NUM = 128
VIDEO_MAX_TOKEN_NUM = 768
FPS = 15.0
FRAME_FACTOR = 2
FPS_MIN_FRAMES = 4
FPS_MAX_FRAMES = 768
MODEL_SEQ_LEN = int(float(os.environ.get('MODEL_SEQ_LEN', 128000)))

# Calculate default pixel values (assuming image_patch_size=16 as used in process_vision_info)
image_patch_size = 16  # This matches what's used in process_vision_info call
image_factor = image_patch_size * SPATIAL_MERGE_SIZE  # = 16 * 2 = 32
VIDEO_FRAME_MIN_PIXELS = VIDEO_MIN_TOKEN_NUM * image_factor * image_factor  # = 128 * 32 * 32 = 131072
VIDEO_FRAME_MAX_PIXELS = VIDEO_MAX_TOKEN_NUM * image_factor * image_factor  # = 768 * 32 * 32 = 786432
DEFAULT_TOTAL_PIXELS = MODEL_SEQ_LEN * image_factor * image_factor * 0.9  # = 128000 * 32 * 32 * 0.9 = 117964800

question_prompt = """
Context: The pool table has a width of 0.9906 and a height of 1.9812. Pockets are marked by colored squares near them. Pocket locations: red at (0, 0), green at (0.9906, 0), orange at (0, 0.9906), blue at (0.9906, 0.9906), gray at (0, 1.9812), and purple at (0.9906, 1.9812). Walls are named by the colors of the two pockets they connect (e.g., the 'red-green' wall is between the red and green pockets). Answer the following question by considering the cue ball (white) movements on the pool table.\nQuestion: What happened in this video?\nA. The first wall hit was purple-grey-wall\nB. The ball was pocketed in the blue pocket\nC. The ball hits 3 different walls\nD. The second wall hit was purple-grey-wall\n\nPlease select the correct option(s). Don't write anything else than the option letter(s). Example: AC.\nHint: There are exactly 1 correct options.
"""

conversations = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": video_path,
                # Frame sampling parameters (choose one approach):
                # Option 1: Use fps-based sampling (default approach)
                "fps": FPS,  # Default: 2.0 fps
                "min_frames": FPS_MIN_FRAMES,  # Default: 4 (rounded to multiple of 2)
                # "max_frames": FPS_MAX_FRAMES,  # Default: min(768, total_frames) - omit to use default
                # Option 2: Use direct frame count (mutually exclusive with fps)
                # "nframes": <int>,  # If set, overrides fps-based sampling
                
                # Time range parameters:
                "video_start": 0.0,  # Default: 0.0 (start from beginning)
                # "video_end": <float>,  # Default: None (use full video) - omit to use full video
                
                # Resizing parameters (optional - defaults are calculated dynamically):
                "min_pixels": 384 * 240,  # Default: 131072 (VIDEO_MIN_TOKEN_NUM * image_factor²)
                "max_pixels": 384 * 240,  # Default: calculated from VIDEO_MAX_TOKEN_NUM and MODEL_SEQ_LEN
                # "total_pixels": DEFAULT_TOTAL_PIXELS,  # Default: 117964800 (MODEL_SEQ_LEN * image_factor² * 0.9)
                # "resized_height": <int>,  # Default: None (calculated from min/max_pixels)
                # "resized_width": <int>,  # Default: None (calculated from min/max_pixels)
                
                # Other parameters (for list of frames input):
                # "sample_fps": FPS,  # Default: 2.0 (only used when video is a list of frames)
                # "raw_fps": <float>,  # Default: same as sample_fps (only used when video is a list of frames)
            },
            {"type": "text", "text": question_prompt},
        ],
    }
]

In [53]:
text = processor.apply_chat_template(
    conversations,
    tokenize=False,
    add_generation_prompt=True,
    padding=True,
)
images, videos, video_kwargs = process_vision_info(
    conversations,
    image_patch_size=16,
    return_video_kwargs=True,
    return_video_metadata=True,
)

/scratch/scottc/causal_pool/.venv/lib/python3.12/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


In [54]:
print(video_kwargs)

{'do_sample_frames': False}


In [55]:
print(text)

<|im_start|>user
<|vision_start|><|video_pad|><|vision_end|>
Context: The pool table has a width of 0.9906 and a height of 1.9812. Pockets are marked by colored squares near them. Pocket locations: red at (0, 0), green at (0.9906, 0), orange at (0, 0.9906), blue at (0.9906, 0.9906), gray at (0, 1.9812), and purple at (0.9906, 1.9812). Walls are named by the colors of the two pockets they connect (e.g., the 'red-green' wall is between the red and green pockets). Answer the following question by considering the cue ball (white) movements on the pool table.
Question: What happened in this video?
A. The first wall hit was purple-grey-wall
B. The ball was pocketed in the blue pocket
C. The ball hits 3 different walls
D. The second wall hit was purple-grey-wall

Please select the correct option(s). Don't write anything else than the option letter(s). Example: AC.
Hint: There are exactly 1 correct options.
<|im_end|>
<|im_start|>assistant



In [56]:
print(len(videos))

1


In [57]:
# split the videos and according metadatas
if videos is not None:
    videos, video_metadatas = zip(*videos)
    videos, video_metadatas = list(videos), list(video_metadatas)
else:
    video_metadatas = None

In [58]:
print(video_metadatas)

[{'fps': 15.0, 'frames_indices': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36,
        37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
        55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70]), 'total_num_frames': 71, 'video_backend': 'torchvision'}]


In [59]:
# since qwen-vl-utils has resize the images/videos,
# we should pass do_resize=False to avoid duplicate operation in processor!
inputs = processor(
    text=text,
    images=images,
    videos=videos,
    video_metadata=video_metadatas,
    return_tensors="pt",
    do_resize=False,
    truncation=False,
    max_length=None,
    padding=True,
    padding_side="left",
    **video_kwargs,
)

In [60]:
for key in inputs:
    print(key, end=" ")
    print(inputs[key].shape)


input_ids torch.Size([1, 3486])
attention_mask torch.Size([1, 3486])
pixel_values_videos torch.Size([11760, 1536])
video_grid_thw torch.Size([1, 3])


In [61]:
from transformers.models.qwen3_vl.video_processing_qwen3_vl

SyntaxError: invalid syntax (3986421020.py, line 1)